<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Descriptive Statistics

In [48]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
n = 80

age = np.clip(np.round(rng.normal(38, 9, size=n)), 18, 70).astype(int)
plan = rng.choice(['Basic', 'Standard', 'Premium'], size=n, p=[0.5, 0.3, 0.2])
mult = pd.Series(plan).map({'Basic': 1.0, 'Standard': 1.8, 'Premium': 3.2}).to_numpy()
spend = np.round(rng.lognormal(mean=8.9, sigma=0.5, size=n) * mult / 10) * 10
tenure = np.clip(np.round((age - 18) * 0.6 + rng.normal(8, 6, size=n)), 1, 60).astype(int)

customers = pd.DataFrame({
    'customer_id': range(3001, 3001 + n),
    'age': age,
    'plan': plan,
    'tenure_months': tenure,
    'monthly_spend': spend,
})

# Capture raw baseline snapshot
customers_raw = customers.copy()



### Notes & Predictions: Part A (Summarise)

* **Mean vs. Median Prediction:** `monthly_spend` will have a mean significantly above its median because it is built from a lognormal distribution multiplied by plan tiers, creating a heavy right tail of high spenders. `age` is generated from a symmetric normal distribution, so its mean and median should sit very close together.
* **Percentage Gap Prediction:** The percentage gap between mean and median will be **much larger for `monthly_spend`** than for `age` due to that extreme right-tail pull.
* **IQR vs. Standard Deviation Prediction:** For `monthly_spend`, the **standard deviation will be larger than the IQR**. Standard deviation squares deviations from the mean, so the extreme values in the right tail inflate it heavily, whereas the IQR measures only the middle 50% of the data.



In [49]:
print("=== PART A: SUMMARISE ===")

# 1. Combined describe()
print("--- 1. Describe: Age and Monthly Spend ---")
display(customers[['age', 'monthly_spend']].describe())

# 2. Mean, Median, Mode, and Percentage Gap
print("\n--- 2. Central Tendency & Percentage Gaps ---")
for col in ['age', 'monthly_spend']:
    mean_val = customers[col].mean()
    med_val = customers[col].median()
    mode_val = customers[col].mode().tolist()
    pct_gap = ((mean_val - med_val) / med_val) * 100
    print(f"{col.upper()} -> Mean: {mean_val:.2f} | Median: {med_val:.2f} | Mode: {mode_val} | Gap: {pct_gap:.1f}%")

# 3. Quantiles and IQR vs Std on monthly_spend
print("\n--- 3. Quantiles and Spread Comparison (Monthly Spend) ---")
quants = customers['monthly_spend'].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
print(quants)

iqr_spend = quants[0.75] - quants[0.25]
std_spend = customers['monthly_spend'].std()
print(f"\nMonthly Spend IQR: {iqr_spend:.2f} vs Standard Deviation: {std_spend:.2f}")
# Result check: IQR 9,852.50 against SD 9,401.74 (n = 80); my prediction was wrong, and the plan mix is a lead to check.

# 4. Testing the plan-mix lead (medians and within-plan IQRs)
print("\n--- 4. Median and IQR of Monthly Spend by Plan ---")
plan_spread = customers.groupby('plan')['monthly_spend'].agg(
    median='median',
    iqr=lambda s: s.quantile(0.75) - s.quantile(0.25)
)
plan_spread['rel_iqr'] = (plan_spread['iqr'] / plan_spread['median']).round(2)
display(plan_spread)

# Note: Normalizing by median removes scale effects (Basic 0.95, Standard 0.59, Premium 0.55).
# The overall IQR (9,852.50) is widened primarily because Q1 (6,945.0) and Q3 (16,797.5) land in different plan tiers.
# Supported: medians differ nearly fourfold across plans (7,010 vs 26,930), consistent with the plan mix widening the middle 50%.
# Comparing within-plan IQRs against the overall IQR (9,852.50) tests whether the within-tier spreads are indeed much smaller.
# Result check: Basic (IQR 6,660.0) and Standard (IQR 7,190.0) both have smaller within-plan spreads than the overall IQR (9,852.50), while Premium is wider (14,717.5).
# This supports the plan-mix explanation: separating Basic (median 7,010.0) and Standard (median 12,290.0) widens the combined middle 50% relative to either majority tier alone.


=== PART A: SUMMARISE ===
--- 1. Describe: Age and Monthly Spend ---


,age,monthly_spend
count,80.000000,80.000000
mean,36.462500,13578.500000
std,7.937962,9401.742513
min,18.000000,2550.000000
25%,31.750000,6945.000000
50%,36.500000,11390.000000
75%,41.000000,16797.500000
max,56.000000,47420.000000



--- 2. Central Tendency & Percentage Gaps ---
AGE -> Mean: 36.46 | Median: 36.50 | Mode: [39] | Gap: -0.1%
MONTHLY_SPEND -> Mean: 13578.50 | Median: 11390.00 | Mode: [11890.0] | Gap: 19.2%

--- 3. Quantiles and Spread Comparison (Monthly Spend) ---
0.10     4203.0
0.25     6945.0
0.50    11390.0
0.75    16797.5
0.90    27844.0
Name: monthly_spend, dtype: float64

Monthly Spend IQR: 9852.50 vs Standard Deviation: 9401.74

--- 4. Median and IQR of Monthly Spend by Plan ---


,median,iqr,rel_iqr
plan,,,
Basic,7010.0,6660.0,0.95
Premium,26930.0,14717.5,0.55
Standard,12290.0,7190.0,0.59


### Notes & Predictions: Part B (Shape & Empirical Rule)

* **Skewness Sign Predictions:**
  * `age`: Near **0** (generated from a symmetric normal distribution).
  * `monthly_spend`: Strongly **positive** (right-skewed due to lognormal generation and tier multipliers).
  * `tenure_months`: Slightly **positive** (clipped at 1 on the left and driven by age, which has a slight right lean above 38 up to 70).
* **Empirical Rule Prediction (68% within 1 SD):** **`age` will be much closer to 68%** because it comes from a normal distribution, whereas `monthly_spend` is heavily right-skewed with an inflated standard deviation that captures a much larger share of the clustered lower values.



In [50]:
print("=== PART B: SHAPE ===")

# 1. Skewness check
n_rows = len(customers)
print(f"Sample size n = {n_rows}")
for col in ['age', 'monthly_spend', 'tenure_months']:
    print(f"{col} skew: {customers[col].skew():.3f}")

# Honest reading lines grounded in the n=80 output:
# - age (n=80, skew=-0.017): Roughly symmetric around the center, closely matching a normal bell curve (68.8% within 1 SD).
# - monthly_spend (n=80, skew=1.393): Strongly right-skewed; 73.8% of values fall within 1 SD, above the 68.3% normal benchmark, because the long right tail widens the SD.
# - tenure_months (n=80, skew=-0.264): Mild negative skew, within normal sampling noise at n = 80, no cause identified (consistent with roughly symmetric verdict).

# 2. Empirical Rule Check (% within 1 standard deviation of the mean)
print("\n--- Empirical Rule Check (Target ~68.3% for Normal) ---")
for col in ['age', 'monthly_spend']:
    m = customers[col].mean()
    s = customers[col].std()
    within_1sd = ((customers[col] >= m - s) & (customers[col] <= m + s)).mean() * 100
    print(f"{col}: {within_1sd:.1f}% of values fall within 1 SD of the mean")


=== PART B: SHAPE ===
Sample size n = 80
age skew: -0.017
monthly_spend skew: 1.393
tenure_months skew: -0.264

--- Empirical Rule Check (Target ~68.3% for Normal) ---
age: 68.8% of values fall within 1 SD of the mean
monthly_spend: 73.8% of values fall within 1 SD of the mean


In [51]:
print("=== PART C: REPORT AND PREVIEW ===")

# 1. Full describe(include='all')
display(customers.describe(include='all'))

# Comment on text-only rows:
# - Rows appearing ONLY for the text column ('plan'): 'unique', 'top', and 'freq'.
# - What 'top' and 'freq' tell us: 'top' shows the most common subscription tier ('Basic'), and 'freq' tells us the exact count of customers on that tier out of n=80.

# 2. Two-Sentence Report on monthly_spend (verified against Part D output):
# "Across n = 80 customers, the typical monthly spend is best represented by the median of 11,390.00 rather than the mean of 13,578.50. The median is the honest choice because strong right-skewness (skew = 1.393) from high-tier spenders pulls the mean 19.2% above the middle customer, overstating typical spend."

# 3. Correlation Matrix
# Prediction: The strongest correlation pair will be 'age' and 'tenure_months', because tenure was explicitly generated using the formula (age - 18) * 0.6 plus noise.
print("\n--- Correlation Matrix ---")
# Drop customer_id first so the code matches our caveat that arbitrary IDs do not belong in correlations
clean_corr = customers.drop(columns='customer_id').select_dtypes('number').corr().round(3)
display(clean_corr)

# Comment on column that does not belong & caution:
# - Column that does not belong: 'customer_id'. It is an arbitrary sequential identifier (3001 to 3080), not a measurable quantity; any correlation with it is pure coincidence.
# - Caution on reading correlations: Correlation measures only linear association between variables and does not prove cause and effect, nor does it capture curved relationships or resist distortion from extreme outliers.



=== PART C: REPORT AND PREVIEW ===


,customer_id,age,plan,tenure_months,monthly_spend
count,80.0000,80.000000,80,80.000000,80.000000
unique,NaN,NaN,3,NaN,NaN
top,NaN,NaN,Basic,NaN,NaN
freq,NaN,NaN,37,NaN,NaN
mean,3040.5000,36.462500,NaN,18.612500,13578.500000
std,23.2379,7.937962,NaN,7.549824,9401.742513
min,3001.0000,18.000000,NaN,1.000000,2550.000000
25%,3020.7500,31.750000,NaN,13.000000,6945.000000
50%,3040.5000,36.500000,NaN,19.000000,11390.000000
75%,3060.2500,41.000000,NaN,25.000000,16797.500000



--- Correlation Matrix ---


,age,tenure_months,monthly_spend
age,1.000,0.623,0.021
tenure_months,0.623,1.000,-0.141
monthly_spend,0.021,-0.141,1.000


In [52]:
print("=== PART D: CHALLENGE FUNCTION ===")

def honest_summary(df, col):
    """
    Prints key descriptive statistics and a data-driven verdict on which
    measure of central tendency to report based on skewness.
    """
    s = df[col].dropna()
    n_count = len(s)
    mean_v = s.mean()
    med_v = s.median()
    modes = s.mode().tolist()
    std_v = s.std()
    iqr_v = s.quantile(0.75) - s.quantile(0.25)
    skew_v = s.skew()

    # Determine verdict based on skewness magnitude
    if skew_v > 0.5:
        verdict = "right skewed: report the median (and IQR)"
    elif skew_v < -0.5:
        verdict = "left skewed: report the median (and IQR)"
    else:
        verdict = "roughly symmetric: mean and median are both reliable"

    print(f"=== HONEST SUMMARY: {col.upper()} (n={n_count}) ===")
    print(f"Mean:   {mean_v:.2f} | Median: {med_v:.2f} | Mode: {modes}")
    print(f"Std:    {std_v:.2f} | IQR:    {iqr_v:.2f} | Skew: {skew_v:.3f}")
    print(f"Verdict -> {verdict}\n")

# Run on all three numeric features
for feature in ['age', 'monthly_spend', 'tenure_months']:
    honest_summary(customers_raw, feature)



=== PART D: CHALLENGE FUNCTION ===
=== HONEST SUMMARY: AGE (n=80) ===
Mean:   36.46 | Median: 36.50 | Mode: [39]
Std:    7.94 | IQR:    9.25 | Skew: -0.017
Verdict -> roughly symmetric: mean and median are both reliable

=== HONEST SUMMARY: MONTHLY_SPEND (n=80) ===
Mean:   13578.50 | Median: 11390.00 | Mode: [11890.0]
Std:    9401.74 | IQR:    9852.50 | Skew: 1.393
Verdict -> right skewed: report the median (and IQR)

=== HONEST SUMMARY: TENURE_MONTHS (n=80) ===
Mean:   18.61 | Median: 19.00 | Mode: [13]
Std:    7.55 | IQR:    12.00 | Skew: -0.264
Verdict -> roughly symmetric: mean and median are both reliable

